# Módulo 3 - Ejercicio 3 (Plantilla)
## Transpilación a Hardware Real, Simulación y Ventaja GPU

### 1. Contexto
El hardware cuántico superconductor real no posee conectividad total. Posee acoplamientos limitados. El compilador cuántico (`transpile`) debe insertar compuertas **SWAP** para conectar qubits no adyacentes, lo que incrementa la profundidad y el número de compuertas CNOT. En presencia de ruido físico, la fidelidad decae.

Por otra parte, al simular circuitos grandes de forma clásica, el vector de estado crece como $2^n$ amplitudes complejas ($16$ bytes cada una). A partir de $24$ a $26$ qubits, un procesador clásico satura su memoria y tiempo de cálculo. Es aquí donde la aceleración por **GPU** en entornos de HPC marca una diferencia.

---

### 2. Referencia Rápida
| Operación | Código |
| :--- | :--- |
| Crear mapa de acoplamiento | `CouplingMap([[0, 1], [1, 2]])` |
| Transpilar con optimización | `transpile(qc, coupling_map=mapa, optimization_level=3)` |
| Consultar profundidad y compuertas | `qc.depth()`, `qc.count_ops()` |
| Modelo de ruido de despolarización | `depolarizing_error(prob, num_qubits)` |
| Simulación en 1 CPU (sin OpenMP) | `AerSimulator(method="statevector", device="CPU", max_parallel_threads=1)` |
| Simulación acelerada por GPU | `AerSimulator(method="statevector", device="GPU")` |


In [1]:
import time
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, transpile
from qiskit.transpiler import CouplingMap
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error

### ️ Tarea 1: Transpilación con Topología Lineal
1. Reconstruye el circuito de Grover de 3 qubits del ejercicio anterior.
2. Define un mapa de acoplamiento lineal: $0 \leftrightarrow 1 \leftrightarrow 2$.
3. Transpila el circuito con `optimization_level=0` y con `optimization_level=3`.
4. Imprime y compara la profundidad y la cantidad de compuertas `cx` en ambos casos.


In [ ]:
# TODO: Construye la función generalizada de Grover y compara transpilación con optimization_level=0 vs 3
def construir_grover(n_qubits: int, objetivo: str = None, iteraciones: int = 1) -> QuantumCircuit:
    """Construye el circuito del Algoritmo de Grover para n cúbits."""
    qc = QuantumCircuit(n_qubits)
    qc.h(range(n_qubits))
    
    if objetivo is None:
        objetivo = '1' * n_qubits
        
    ultimo = n_qubits - 1
    controles = list(range(ultimo))
    
    for _ in range(iteraciones):
        # Oráculo
        for q, bit in enumerate(reversed(objetivo)):
            if bit == '0':
                qc.x(q)
        qc.h(ultimo)
        qc.mcx(controles, ultimo)
        qc.h(ultimo)
        for q, bit in enumerate(reversed(objetivo)):
            if bit == '0':
                qc.x(q)
                
        # Difusor
        qc.h(range(n_qubits))
        qc.x(range(n_qubits))
        qc.h(ultimo)
        qc.mcx(controles, ultimo)
        qc.h(ultimo)
        qc.x(range(n_qubits))
        qc.h(range(n_qubits))
        
    qc.measure_all()
    return qc

# 1. Instancia el circuito para 3 cúbits, objetivo |110> y 2 iteraciones:
# qc_base = construir_grover(3, objetivo="110", iteraciones=2)
# mapa_lineal = CouplingMap([[0, 1], [1, 0], [1, 2], [2, 1]])

### TU CÓDIGO AQUÍ ###
# compuertas_base = ['cx', 'id', 'rz', 'sx', 'x']
# t_opt0 = transpile(qc_base, coupling_map=mapa_lineal, basis_gates=compuertas_base, optimization_level=0, seed_transpiler=42)
# t_opt3 = transpile(qc_base, coupling_map=mapa_lineal, basis_gates=compuertas_base, optimization_level=3, seed_transpiler=42)


### ️ Tarea 2: Creación de Modelo de Ruido y Simulación
1. Crea un `NoiseModel` que incluya:
   - Error de despolarización de $2.0\%$ en compuertas de 2 qubits (`cx`).
2. Instancia `AerSimulator(noise_model=tu_modelo)`.
3. Ejecuta el circuito transpilado (`optimization_level=3`) con 2048 disparos.
4. Calcula el porcentaje de éxito del estado $|110\rangle$ y compáralo con la simulación ideal.


In [ ]:
# TODO: Simulación con ruido
### TU CÓDIGO AQUÍ ### # nm = NoiseModel()
# ...
# backend_ruidoso = AerSimulator(noise_model=nm)
# counts = backend_ruidoso.run(t_qc, shots=2048).result().get_counts()
# print(counts)


### ️ Tarea 3 (Conexión HPC): Medición en Vivo de CPU (1 solo núcleo) vs. GPU NVIDIA A100 en el Clúster

En esta tarea mediremos en tiempo real sobre el sistema los tiempos exactos de simulación de circuitos cuánticos para comparar:
- **CPU Clásico:** `AerSimulator(method="statevector", device="CPU", max_parallel_threads=1)`
- **GPU de Supercómputo (NVIDIA A100 / cuQuantum):** `AerSimulator(method="statevector", device="GPU")`

> **Nota técnica sobre el entorno GPU:** El paquete estándar `qiskit-aer` viene compilado únicamente para CPU. Para habilitar la GPU, se debe instalar el paquete con binarios CUDA correspondientes mediante:
> `pip install qiskit-aer-gpu-cu11` (y reiniciar el kernel de Jupyter).

#### Instrucciones:
1. Ejecuta este notebook en un nodo del clúster equipado con GPU (o en Google Colab con entorno GPU A100 o T4).
2. Corre el bucle de medición en vivo que evalúa el mismo circuito cuántico para distintos tamaños ($n \in [16, 18, 20, 22, 24, 26]$ qubits).
3. A partir de los vectores reales medidos (`tiempos_cpu` y `tiempos_gpu`), calcula el factor de aceleración ($t_{\text{CPU}} / t_{\text{GPU}}$) y genera los gráficos comparativos.

In [ ]:
# Configuración de simuladores para la comparativa CPU vs GPU
# CPU: 1 solo núcleo (sin paralelismo OpenMP)
sim_cpu1 = AerSimulator(method="statevector", device="CPU", max_parallel_threads=1)

# GPU:
try:
    sim_gpu = AerSimulator(method="statevector", device="GPU")
    # Verificación de ejecución real en el backend C++/CUDA
    sim_gpu.run(QuantumCircuit(2)).result()
    gpu_disponible = True
    print(" GPU NVIDIA detectada y activa para mediciones en vivo.")
except Exception as e:
    gpu_disponible = False
    print(f"Aviso: GPU no activa en este entorno ({e}).")
    print("Para habilitar la GPU:")
    print("  1. Instala el paquete con soporte CUDA: !pip install qiskit-aer-gpu-cu11")
    print("  2. Reinicia el Kernel de Jupyter (Kernel -> Restart Kernel)")
    print("  3. Verifica con AerSimulator().available_devices() que aparezca ('CPU', 'GPU')")


In [ ]:
# 2. Medición en tiempo real sobre el sistema: Grover a gran escala (CPU 1 núcleo vs GPU)
qubits_eval = [16, 18, 20, 22, 24, 26]
tiempos_cpu = []
tiempos_gpu = []

print(f"{'Qubits':<8}{'Amplitudes':<15}{'Tiempo CPU (1 núcleo)':<24}{'Tiempo GPU A100':<20}{'Aceleración':<10}")
print("=" * 77)

for n in qubits_eval:
    # Evaluamos el Algoritmo de Grover en dimensión n con 1 iteración
    qc = construir_grover(n, iteraciones=1)
    
    # Medición en CPU (1 solo núcleo)
    t0 = time.time()
    sim_cpu1.run(qc, shots=500).result()
    t_cpu = time.time() - t0
    tiempos_cpu.append(t_cpu)
    
    # Medición en GPU
    if gpu_disponible:
        t0 = time.time()
        sim_gpu.run(qc, shots=500).result()
        t_gpu = time.time() - t0
        tiempos_gpu.append(t_gpu)
        sp = t_cpu / t_gpu
        sp_str = f"{sp:5.1f}x"
        t_gpu_str = f"{t_gpu:6.3f} s"
    else:
        t_gpu_str = "No disponible"
        sp_str = "N/A"
        
    print(f"{n:<8}{2**n:<15,d}{t_cpu:6.3f} s{' '*16}{t_gpu_str:<20}{sp_str:<10}")

In [ ]:
# 3. TODO: Visualización de los datos reales medidos en el sistema
# A partir de los vectores medidos en tiempo real (tiempos_cpu y tiempos_gpu):
# 1. Calcula: speedup = np.array(tiempos_cpu) / np.array(tiempos_gpu)
# 2. Genera los gráficos en dos paneles con matplotlib:
# - Panel 1: Tiempos de simulación en escala logarítmica (CPU 1 núcleo vs GPU A100)
# - Panel 2: Factor de aceleración con gráfico de barras

### TU CÓDIGO AQUÍ ### # fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5), dpi=100)
# ...
# plt.show()